# NOMOPHOBIA CPU Research Suite

**Purpose:** resolve the highest-value pre-S3 questions in one synchronous CPU notebook using cheap gates before expensive experiments.

### Required Kaggle input
- Official competition input containing `train.csv`, `test.csv`, and `sample_submission.csv`.

### Optional Kaggle input
- The public ~7,500-row source dataset CSV containing the 12 raw predictors plus `addicted_label`. If absent, the source-row augmentation arm auto-skips.

### Notebook settings
- **Accelerator:** None (CPU)
- **Internet:** **ON** for the simplest setup, used only to clone the GitHub repository. If Internet is OFF, attach the provided `nomophobia-source.zip` package as an input and the bootstrap cell will use it instead.
- **Persistence:** optional. The suite writes all outputs under `/kaggle/working/nomophobia_cpu_research`.

The default `balanced` profile is tuned for CPU. Use `quick` to validate plumbing or `thorough` when you are willing to spend substantially more CPU for mature S1-scale screens.

In [ ]:
from pathlib import Path
import os, json, multiprocessing

PROFILE = "balanced"      # quick | balanced | thorough
REPO_REF = "main"
SEED = 20260816
THREADS = max(1, multiprocessing.cpu_count())

DATA_DIR = Path("/kaggle/input/playground-series-s6e8")
OUT_ROOT = Path("/kaggle/working/nomophobia_cpu_research")
REPO_DIR = Path("/kaggle/working/nomophobia")

print("CPU threads:", THREADS)
print("Data dir:", DATA_DIR)
print("Output dir:", OUT_ROOT)
assert (DATA_DIR / "train.csv").exists(), "Attach the Playground Series S6E8 competition input."


## Bootstrap the audited repository

The cell prefers GitHub when Internet is enabled. If cloning fails, it searches attached Kaggle inputs for `nomophobia-source.zip` or a directory containing `pyproject.toml` and `src/s6e8`.

In [ ]:
import subprocess, shutil, zipfile

def find_offline_repo():
    for candidate in Path("/kaggle/input").rglob("pyproject.toml"):
        parent = candidate.parent
        if (parent / "src" / "s6e8").exists():
            return parent
    for z in Path("/kaggle/input").rglob("nomophobia-source.zip"):
        target = Path("/kaggle/working/nomophobia_offline")
        if target.exists():
            shutil.rmtree(target)
        target.mkdir(parents=True)
        with zipfile.ZipFile(z) as f:
            f.extractall(target)
        for candidate in target.rglob("pyproject.toml"):
            if (candidate.parent / "src" / "s6e8").exists():
                return candidate.parent
    return None

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

try:
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", REPO_REF,
        "https://github.com/sidhulyalkar/nomophobia.git", str(REPO_DIR)
    ], check=True)
except Exception as exc:
    print("Git clone unavailable:", exc)
    offline = find_offline_repo()
    if offline is None:
        raise RuntimeError("Could not obtain repository code. Enable Internet or attach nomophobia-source.zip.")
    shutil.copytree(offline, REPO_DIR)

subprocess.run([os.sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps"], check=True)
subprocess.run(["nomophobia", "validate", "--data-dir", str(DATA_DIR)], check=True)
print("Repository ready:", REPO_DIR)


## Auto-detect the optional source CSV

If several external CSVs are attached, this cell picks the first one outside the competition directory that contains `addicted_label` plus the expected smartphone predictors. You can override `ORIGINAL_CSV` manually.

In [ ]:
import pandas as pd

ORIGINAL_CSV = None
required = {"addicted_label", "daily_screen_time_hours", "social_media_hours", "gaming_hours"}
for path in Path("/kaggle/input").rglob("*.csv"):
    if str(path).startswith(str(DATA_DIR)):
        continue
    try:
        cols = {str(c).strip().lower() for c in pd.read_csv(path, nrows=3).columns}
    except Exception:
        continue
    if required.issubset(cols):
        ORIGINAL_CSV = path
        break

print("Optional source CSV:", ORIGINAL_CSV or "not attached; source augmentation will skip")


## Run the gated CPU suite

Execution order:
1. transductive frequency + source-safety gate,
2. marginal frequency-family decomposition,
3. capacity/diversity curve,
4. higher-order density geometry only if safety survives,
5. optional low-weight source-row augmentation.

The runner hashes competition inputs once, uses `--no-hash-inputs` for child experiments, and creates a single ZIP for easy chaining.

In [ ]:
cmd = [
    os.sys.executable,
    str(REPO_DIR / "scripts" / "run_cpu_research_suite.py"),
    "--data-dir", str(DATA_DIR),
    "--out-root", str(OUT_ROOT),
    "--profile", PROFILE,
    "--seed", str(SEED),
    "--threads", str(THREADS),
]
if ORIGINAL_CSV is not None:
    cmd += ["--original-csv", str(ORIGINAL_CSV)]

print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


## Decision summary

The important artifact is `cpu_research_decision.json`. The ZIP contains the decision plus every experiment JSON/manifest produced in this run.

In [ ]:
decision = json.loads((OUT_ROOT / "cpu_research_decision.json").read_text())
print(json.dumps({
    "profile": decision["profile"],
    "frequency_safety": decision["frequency_safety"],
    "capacity_route": decision["capacity_route"],
    "geometry_advanced_arms": decision["geometry_advanced_arms"],
    "source_advanced_weights": decision["source_advanced_weights"],
    "recommended_next_step": decision["recommended_next_step"],
    "elapsed_minutes": round(decision["elapsed_seconds"] / 60, 1),
}, indent=2))

print("\nFiles to preserve:")
for p in sorted(Path("/kaggle/working").glob("nomophobia_cpu_research*")):
    print(" ", p)


### After the run

Use **Save Version** with output files enabled. For Notebook 2, add this notebook's saved output as an input. Notebook 2 will auto-discover `cpu_research_decision.json`, but it does not require it to run the authoritative baseline if you intentionally skip Notebook 1.